# MiniMax-H3 NF4をGoogle Colabで動かす

MiniMax-H3の **NF4量子化版** を、DiffSynth-Studioのlow-VRAM inferenceを使ってGoogle Colab上で動かします。

使用モデル:

```text
DiffSynth-Studio/MiniMax-H3-NF4
```

MiniMax-H3は、テキストから **動画＋ステレオ音声** を同時生成できるomni-modal video generation modelです。

このNotebookでは、書籍のColabプログラムとできるだけ同じ流れにして、

```text
実行環境の準備
→ Google Drive / cache
→ DiffSynth-Studioの準備
→ NF4モデル読み込み
→ 最小Text-to-Video+Audio生成
→ 結果表示
```

という構成にしています。

> **重要**
>
> 公式DiffSynth-StudioではNF4版＋VRAM Managementにより最低7GB VRAMでの推論を案内しています。
> ただし、Google Colab無料版T4ではGPU VRAMだけでなくCPU RAM・ディスクI/O・生成時間も制約になります。
>
> このNotebookは、まず「短い・低解像度・少ないstep」で1本生成できるかを確認する実験版です。


In [ ]:
# =========================================
# コード1 実行環境の準備
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V

import torch
import os
import shutil

if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを有効にしてください。")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

props = torch.cuda.get_device_properties(0)
print("GPU memory: %.1f GB" % (props.total_memory / 1024**3))

usage = shutil.disk_usage("/")
print("Root disk free: %.1f GB" % (usage.free / 1024**3))

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("COMPUTE_DTYPE:", COMPUTE_DTYPE)


In [ ]:
# =========================================
# コード2 Google Driveとキャッシュ設定
# =========================================
from google.colab import drive
from pathlib import Path
import os
import shutil

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
USE_DRIVE_CACHE = True

if USE_DRIVE_CACHE:
    CACHE_DIR = PROJECT_DIR / "Program" / "modelscope_cache_minimax_h3"
else:
    CACHE_DIR = Path("/content/modelscope_cache_minimax_h3")

CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["MODELSCOPE_CACHE"] = str(CACHE_DIR)
os.environ["MODELSCOPE_ENDPOINT"] = "https://modelscope.ai"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

usage = shutil.disk_usage(CACHE_DIR)

print("CACHE_DIR:", CACHE_DIR)
print("free space: %.1f GB" % (usage.free / 1024**3))

if usage.free < 45 * 1024**3:
    print("WARNING: 45GB以上の空き容量を推奨します。")


In [ ]:
# =========================================
# コード3 DiffSynth-Studioをインストール
# =========================================
%cd /content

!rm -rf DiffSynth-Studio
!git clone -q https://github.com/modelscope/DiffSynth-Studio.git

%cd /content/DiffSynth-Studio

# DiffSynth-Studio本体
%pip -q install -e .

# MiniMax-H3 NF4で追加使用するライブラリ
%pip -q install -U \
    modelscope \
    av \
    bitsandbytes

print("DiffSynth-Studio installed.")

## Offload mode

DiffSynth-Studio公式にはCPU offloadと、さらに省メモリなdisk offloadがあります。

無料版ColabではCPU RAMも小さいため、初期値は **disk offload** にしています。

有料版などCPU RAMに余裕がある場合は、

```python
OFFLOAD_MODE = "cpu"
```

へ変更できます。


In [ ]:
# =========================================
# コード4 MiniMax-H3 NF4モデルを読み込む
# =========================================
import torch

from diffsynth.pipelines.minimax_h3_audio_video import (
    MiniMaxH3Pipeline,
    ModelConfig,
)
from diffsynth.utils.data.audio_video import write_video_audio

MODEL_ID = "DiffSynth-Studio/MiniMax-H3-NF4"
PROCESSOR_ID = "MiniMax/MiniMax-H3"

OFFLOAD_MODE = "disk"

if OFFLOAD_MODE == "disk":
    offload_dtype = "disk"
    offload_device = "disk"
else:
    offload_dtype = COMPUTE_DTYPE
    offload_device = "cpu"

vram_config = {
    "offload_dtype": offload_dtype,
    "offload_device": offload_device,
    "onload_dtype": COMPUTE_DTYPE,
    "onload_device": "cpu",
    "preparing_dtype": COMPUTE_DTYPE,
    "preparing_device": "cuda",
    "computation_dtype": COMPUTE_DTYPE,
    "computation_device": "cuda",
}

total_vram_gb = torch.cuda.mem_get_info("cuda")[1] / (1024**3)
VRAM_LIMIT_GB = max(6.0, total_vram_gb - 2.0)

print("OFFLOAD_MODE:", OFFLOAD_MODE)
print("VRAM_LIMIT_GB:", VRAM_LIMIT_GB)
print("Loading MiniMax-H3 NF4...")

pipe = MiniMaxH3Pipeline.from_pretrained(
    torch_dtype=COMPUTE_DTYPE,
    device="cuda",
    model_configs=[
        ModelConfig(
            model_id=MODEL_ID,
            origin_file_pattern="minimax-h3-fl2va-nf4.safetensors",
            **vram_config,
        ),
        ModelConfig(
            model_id=MODEL_ID,
            origin_file_pattern="minimax-h3-text-encoder-nf4.safetensors",
            **vram_config,
        ),
        ModelConfig(
            model_id=MODEL_ID,
            origin_file_pattern="video_vae_nf4.safetensors",
            **vram_config,
        ),
        ModelConfig(
            model_id=MODEL_ID,
            origin_file_pattern="audio_vae_nf4.safetensors",
            **vram_config,
        ),
    ],
    processor_config=ModelConfig(
        model_id=PROCESSOR_ID,
        origin_file_pattern="FL2VA/processor/",
    ),
    vram_limit=VRAM_LIMIT_GB,
)

print("Pipeline loaded.")
print("GPU allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))
print("GPU reserved : %.2f GB" % (torch.cuda.memory_reserved() / 1024**3))


## 最小生成テスト

公式例は480×832、124 frames、50 inference stepsです。

無料版T4ではかなり重いため、このNotebookでは最初に、

```text
256 × 448
56 frames
12 inference steps
```

で試します。

品質確認ではなく、まず最後まで1本生成できるかを確認するための設定です。


In [ ]:
# =========================================
# コード5 Text -> Video + Audio 最小生成テスト
# =========================================
from pathlib import Path
import time
import torch

OUTPUT_DIR = Path("/content/minimax_h3_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "minimax_h3_nf4_test.mp4"

prompt = (
    "A small red robot walks slowly across a quiet futuristic laboratory. "
    "Soft cinematic lighting. The robot says in English: "
    "\"Hello from MiniMax H3.\""
)

HEIGHT = 256
WIDTH = 448
NUM_FRAMES = 56
NUM_INFERENCE_STEPS = 12
SEED = 0

print("Prompt:", prompt)
print(f"Generating {WIDTH}x{HEIGHT}, {NUM_FRAMES} frames, {NUM_INFERENCE_STEPS} steps...")

start = time.time()

with torch.inference_mode():
    video, audio = pipe(
        prompt=prompt,
        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        num_inference_steps=NUM_INFERENCE_STEPS,
        seed=SEED,
        tiled=True,
    )

elapsed = time.time() - start

write_video_audio(
    video=video,
    audio=audio,
    output_path=str(OUTPUT_PATH),
    fps=24,
    audio_sample_rate=32000,
)

print("Generated:", OUTPUT_PATH)
print("Elapsed: %.1f sec (%.1f min)" % (elapsed, elapsed / 60))
print("GPU allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))
print("GPU reserved : %.2f GB" % (torch.cuda.memory_reserved() / 1024**3))


In [ ]:
# =========================================
# コード6 生成動画を表示
# =========================================
from IPython.display import Video, display

display(Video(str(OUTPUT_PATH), embed=True, width=640))


## 公式設定へ近づける場合

最小生成が成功した後で、段階的に条件を上げます。

```python
HEIGHT = 320
WIDTH = 576
NUM_FRAMES = 56
NUM_INFERENCE_STEPS = 20
```

次に、

```python
HEIGHT = 480
WIDTH = 832
NUM_FRAMES = 124
NUM_INFERENCE_STEPS = 20
```

最終的な公式Quick Startは、

```python
HEIGHT = 480
WIDTH = 832
NUM_FRAMES = 124
NUM_INFERENCE_STEPS = 50
```

です。

### 注意

- 初回ダウンロードは非常に大きいです。
- Disk offloadは省メモリですが、かなり遅くなります。
- T4ではBF16ではなくFP16へフォールバックします。これはColab T4向けの実験的変更です。


In [ ]:
# =========================================
# コード5 Text -> Video + Audio 事前生成テスト2
# =========================================
from pathlib import Path
import time
import torch

OUTPUT_DIR = Path("/content/minimax_h3_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "minimax_h3_nf4_test2.mp4"

prompt = (
    "A small red robot walks slowly across a quiet futuristic laboratory. "
    "Soft cinematic lighting. The robot says in English: "
    "\"Hello from MiniMax H3.\""
)

HEIGHT = 320
WIDTH = 576
NUM_FRAMES = 56
NUM_INFERENCE_STEPS = 20
SEED = 0

print("Prompt:", prompt)
print(f"Generating {WIDTH}x{HEIGHT}, {NUM_FRAMES} frames, {NUM_INFERENCE_STEPS} steps...")

start = time.time()

with torch.inference_mode():
    video, audio = pipe(
        prompt=prompt,
        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        num_inference_steps=NUM_INFERENCE_STEPS,
        seed=SEED,
        tiled=True,
    )

elapsed = time.time() - start

write_video_audio(
    video=video,
    audio=audio,
    output_path=str(OUTPUT_PATH),
    fps=24,
    audio_sample_rate=32000,
)

print("Generated:", OUTPUT_PATH)
print("Elapsed: %.1f sec (%.1f min)" % (elapsed, elapsed / 60))
print("GPU allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))
print("GPU reserved : %.2f GB" % (torch.cuda.memory_reserved() / 1024**3))


In [ ]:
# =========================================
# コード6 生成動画を表示
# =========================================
from IPython.display import Video, display

display(Video(str(OUTPUT_PATH), embed=True, width=640))

In [ ]:
# =========================================
# コード5 Text -> Video + Audio 事前生成テスト3
# =========================================
from pathlib import Path
import time
import torch

OUTPUT_DIR = Path("/content/minimax_h3_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "minimax_h3_nf4_test3.mp4"

prompt = (
    "A small red robot walks slowly across a quiet futuristic laboratory. "
    "Soft cinematic lighting. The robot says in English: "
    "\"Hello from MiniMax H3.\""
)

HEIGHT = 480
WIDTH = 832
NUM_FRAMES = 124
NUM_INFERENCE_STEPS = 20
SEED = 0

print("Prompt:", prompt)
print(f"Generating {WIDTH}x{HEIGHT}, {NUM_FRAMES} frames, {NUM_INFERENCE_STEPS} steps...")

start = time.time()

with torch.inference_mode():
    video, audio = pipe(
        prompt=prompt,
        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        num_inference_steps=NUM_INFERENCE_STEPS,
        seed=SEED,
        tiled=True,
    )

elapsed = time.time() - start

write_video_audio(
    video=video,
    audio=audio,
    output_path=str(OUTPUT_PATH),
    fps=24,
    audio_sample_rate=32000,
)

print("Generated:", OUTPUT_PATH)
print("Elapsed: %.1f sec (%.1f min)" % (elapsed, elapsed / 60))
print("GPU allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))
print("GPU reserved : %.2f GB" % (torch.cuda.memory_reserved() / 1024**3))

In [ ]:
# =========================================
# コード6 生成動画を表示
# =========================================
from IPython.display import Video, display

display(Video(str(OUTPUT_PATH), embed=True, width=640))

In [ ]:
# =========================================
# コード5 Text -> Video + Audio 公式Quick Startテスト
# =========================================
from pathlib import Path
import time
import torch

OUTPUT_DIR = Path("/content/minimax_h3_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "minimax_h3_nf4_test3.mp4"

prompt = (
    "A small red robot walks slowly across a quiet futuristic laboratory. "
    "Soft cinematic lighting. The robot says in English: "
    "\"Hello from MiniMax H3.\""
)

HEIGHT = 480
WIDTH = 832
NUM_FRAMES = 124
NUM_INFERENCE_STEPS = 50
SEED = 0

print("Prompt:", prompt)
print(f"Generating {WIDTH}x{HEIGHT}, {NUM_FRAMES} frames, {NUM_INFERENCE_STEPS} steps...")

start = time.time()

with torch.inference_mode():
    video, audio = pipe(
        prompt=prompt,
        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        num_inference_steps=NUM_INFERENCE_STEPS,
        seed=SEED,
        tiled=True,
    )

elapsed = time.time() - start

write_video_audio(
    video=video,
    audio=audio,
    output_path=str(OUTPUT_PATH),
    fps=24,
    audio_sample_rate=32000,
)

print("Generated:", OUTPUT_PATH)
print("Elapsed: %.1f sec (%.1f min)" % (elapsed, elapsed / 60))
print("GPU allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))
print("GPU reserved : %.2f GB" % (torch.cuda.memory_reserved() / 1024**3))

In [ ]:
# =========================================
# コード6 生成動画を表示
# =========================================
from IPython.display import Video, display

display(Video(str(OUTPUT_PATH), embed=True, width=640))

In [ ]:
# =========================================
# 生成後のGPUメモリ解放
# =========================================
import gc
import torch

del video
del audio

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated: %.2f GB"
    % (torch.cuda.memory_allocated() / 1024**3)
)
print(
    "GPU reserved : %.2f GB"
    % (torch.cuda.memory_reserved() / 1024**3)
)

# 複雑のプロンプト

In [ ]:
# =========================================
# コード5 Text -> Video + Audio 公式Quick Startテスト
# =========================================
from pathlib import Path
import time
import torch

OUTPUT_DIR = Path("/content/minimax_h3_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "minimax_h3_nf4_test3.mp4"

prompt = ("""
A small white robot is standing in a futuristic laboratory.
The camera is fixed.
The robot looks directly at the camera and says clearly:
"Hello. Let's begin the experiment."
Quiet electronic sounds can be heard in the background.
""")

HEIGHT = 480
WIDTH = 832
NUM_FRAMES = 124
NUM_INFERENCE_STEPS = 50
SEED = 0

print("Prompt:", prompt)
print(f"Generating {WIDTH}x{HEIGHT}, {NUM_FRAMES} frames, {NUM_INFERENCE_STEPS} steps...")

start = time.time()

with torch.inference_mode():
    video, audio = pipe(
        prompt=prompt,
        height=HEIGHT,
        width=WIDTH,
        num_frames=NUM_FRAMES,
        num_inference_steps=NUM_INFERENCE_STEPS,
        seed=SEED,
        tiled=True,
    )

elapsed = time.time() - start

write_video_audio(
    video=video,
    audio=audio,
    output_path=str(OUTPUT_PATH),
    fps=24,
    audio_sample_rate=32000,
)

print("Generated:", OUTPUT_PATH)
print("Elapsed: %.1f sec (%.1f min)" % (elapsed, elapsed / 60))
print("GPU allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))
print("GPU reserved : %.2f GB" % (torch.cuda.memory_reserved() / 1024**3))

In [ ]:
# =========================================
# コード6 生成動画を表示
# =========================================
from IPython.display import Video, display

display(Video(str(OUTPUT_PATH), embed=True, width=640))